# Unsupervised Learning Trading Strategy

* Download/Load NSE stocks prices data.
* Calculate different features and indicators on each stock.
* Aggregate on monthly level and filter top 150 most liquid stocks.
* Calculate Monthly Returns for different time-horizons.
* Download Fama-French Factors and Calculate Rolling Factor Betas.
* For each month fit a K-Means Clustering Algorithm to group similar assets based on their features.
* For each month select assets based on the cluster and form a portfolio based on Efficient Frontier max sharpe ratio optimization.
* Visualize Portfolio returns and compare to NSE returns.

# All Packages Needed:
* pandas, numpy, matplotlib, statsmodels, pandas_datareader, datetime, yfinance, sklearn, PyPortfolioOpt

## 1. Download/Load SP500 stocks prices data.

In [1]:
%%bash
pip install "pandasai>=3.0.0b2"
pip install TA-Lib

In [2]:
%%bash
sudo apt install libtool -y
git clone https://github.com/ta-lib/ta-lib/
cd ta-lib
chmod +x autogen.sh  # ensure the permissions are set to generate the configure file
./autogen.sh         # generate the configure file
./configure
make
sudo make install


Reading package lists...
Building dependency tree...
Reading state information...
libtool is already the newest version (2.4.6-15build2).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.
aclocal
autoheader
libtoolize --copy --force
libtoolize: putting auxiliary files in '.'.
libtoolize: copying file './ltmain.sh'
libtoolize: putting macros in AC_CONFIG_MACRO_DIRS, 'm4'.
libtoolize: copying file 'm4/libtool.m4'
libtoolize: copying file 'm4/ltoptions.m4'
libtoolize: copying file 'm4/ltsugar.m4'
libtoolize: copying file 'm4/ltversion.m4'
libtoolize: copying file 'm4/lt~obsolete.m4'
automake -a -c
autoconf
checking for a BSD-compatible install... /usr/bin/install -c
checking whether build environment is sane... yes
checking for a race-free mkdir -p... /usr/bin/mkdir -p
checking for gawk... no
checking for mawk... mawk
checking whether make sets $(MAKE)... yes
checking whether make supports nested variables... yes
checking whether make supports the include directive... yes (G



fatal: destination path 'ta-lib' already exists and is not an empty directory.
./autogen.sh: 6: [[: not found
configure.ac:13: installing './compile'
configure.ac:10: installing './missing'
src/ta_abstract/Makefile.am: installing './depcomp'
/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.1

In [3]:
from statsmodels.regression.rolling import RollingOLS
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np
import datetime as dt
import requests
import pandas as pd
import pandasai
import talib
import warnings
warnings.filterwarnings('ignore')
# Set pandas display options to show full output
pd.set_option('display.max_rows', None)  # Display all rows
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', None)  # No wrapping (adjust this as per your terminal/console width)
pd.set_option('display.max_colwidth', None)  # Don't truncate

In [4]:
# Define the cookies as a dictionary
cookies = {

}

# Define the URL
url = 'https://live.mystocks.co.ke/rmw.php'


In [5]:
# Function to fetch the data
def fetch_data(url, cookies):
    """
    Fetches the data from the provided URL and returns the raw table.
    """
    # Send a GET request with cookies
    response = requests.get(url, cookies=cookies)

    if response.status_code == 200:
        # Parse the HTML content with pandas read_html (this extracts all tables)
        tables = pd.read_html(response.text)

        # Assuming the market data is in table 7
        market_data = tables[7]
        return market_data
    else:
        print(f"Failed to fetch the page. Status code: {response.status_code}")
        return None

In [6]:
# Function to wrangle the data
def wrangle_data(market_data):
    """
    Clean and process the fetched data into a usable format.
    """
    # Drop the first column (the button column) and reset the index
    market_data_clean = market_data.drop(columns=[0]).reset_index(drop=True)

    # Rename the columns to match the correct ones
    market_data_clean.columns = [
        "Security", "Open", "Closing", "Change", "Change.1", "High", "Low", "Volume", "VWAP", "Deals", "Turnover", "Foreign", "Time"
    ]

    # Clean the data further by removing rows with NaN values in all columns
    market_data_clean = market_data_clean.dropna(how='all')

    # Create a DataFrame for the cleaned data
    df = pd.DataFrame(market_data_clean)

    # Convert 'foreign' column to numeric values (handle special cases like percentage and 'M' values)
    def process_foreign(value):
        if value == '-' or value == 'NaN' or pd.isna(value):
            return np.nan
        elif isinstance(value, str) and '%' in value:
            return float(value.replace('%', '')) / 100
        elif isinstance(value, str) and 'M' in value:
            return float(value.replace('M', '').replace(',', '')) * 1e6
        elif isinstance(value, (int, float)):
            return value
        return np.nan

    # Apply the 'process_foreign' function to the 'foreign' column
    df['Foreign'] = df['Foreign'].apply(process_foreign)

    # Convert 'Time' to datetime
    df['Time'] = pd.to_datetime(df['Time'], errors='coerce')

    # Convert numeric columns to floats
    numeric_columns = ['High', 'Low', 'Closing', 'Open', 'Volume', 'VWAP', 'Deals', 'Turnover']
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Set both 'Time' and 'Security' as multi-index
    df.set_index(['Time', 'Security'], inplace=True)
    df.columns = df.columns.str.lower()

    return df

In [7]:
# Fetch the raw data from the URL
raw_data = fetch_data(url, cookies)

# If data was fetched, wrangle it
if raw_data is not None:
    cleaned_data = wrangle_data(raw_data)

In [8]:
cleaned_data
df = cleaned_data.copy()

In [9]:
df.columns

Index(['open', 'closing', 'change', 'change.1', 'high', 'low', 'volume',
       'vwap', 'deals', 'turnover', 'foreign'],
      dtype='object')

## 2. Calculate features and technical indicators for each stock.

* Garman-Klass Volatility
* RSI
* Bollinger Bands
* ATR
* MACD
* Dollar Volume

\begin{equation}
\text{Garman-Klass Volatility} = \frac{(\ln(\text{High}) - \ln(\text{Low}))^2}{2} - \left(2\ln(2) - 1\right) \left(\ln(\text{Closing}) - \ln(\text{Open})\right)^2
\end{equation}


In [22]:
# Garman-Klass Volatility using TA-Lib (assuming `high`, `low`, `close`, `open` are present)
df['garman_klass_vol'] = ((np.log(df['high']) - np.log(df['low']))**2)/2 - (2 * np.log(2) - 1) * ((np.log(df['closing']) - np.log(df['open']))**2)

# RSI using TA-Lib (length = 20)
df['rsi'] = talib.RSI(df['closing'], timeperiod=20)

# Bollinger Bands using TA-Lib (length = 20, 2 standard deviations for upper and lower bands)
df['bb_low'], df['bb_mid'], df['bb_high'] = talib.BBANDS(df['closing'], timeperiod=20, nbdevup=2, nbdevdn=2, matype=0)

# ATR using TA-Lib (length = 14)
df['atr'] = talib.ATR(df['high'], df['low'], df['closing'], timeperiod=14)

# MACD using TA-Lib (MACD, Signal, Hist)
macd, macdsignal, macdhist = talib.MACD(df['closing'], fastperiod=12, slowperiod=26, signalperiod=9)
df['macd'] = macd

# Dollar Volume (no change, same calculation as before)
df['dollar_volume'] = (df['closing'] * df['volume']) / 1e6

In [23]:

df

open  closing change change.1    high  \
Time                Security                                             
2025-07-19 15:25:06 ABSA        19.70    19.55   0.15    0.76%   19.80   
                    AMAC        56.00    56.00      -        -     NaN   
2025-07-19 15:25:05 ARM          5.55     5.55      -        -     NaN   
                    BAMB        47.20    47.20      -        -     NaN   
                    BAT        381.00   381.00      -        -  385.00   
                    BKG         35.30    35.05   0.25    0.71%   35.25   
                    BOC         89.00    90.75   1.75    1.97%   90.75   
                    BRIT         8.14     8.28   0.14    1.72%    8.34   
                    CABL         1.00     1.00      -        -     NaN   
                    CARB        21.30    21.90   0.60    2.82%   22.00   
2025-07-19 15:25:06 CGEN        23.35    23.35      -        -   23.50   
2025-07-19 15:25:05 CIC          3.21     3.33   0.12    3.74%    3.42   
2025-07-19 15:25:06 COOP        16.85    16.75   0.10    0.59%   16.85   
                    CRWN        40.00    41.00   1.00    2.50%   41.00   
                    CTUM        11.80    11.85   0.05    0.42%   12.00   
2025-07-19 15:25:05 DCON         0.45     0.45      -        -     NaN   
                    DTK         78.00    78.25   0.25    0.32%   80.00   
                    EABL       199.50   194.25   5.25    2.63%  202.50   
                    EGAD        11.50    11.50      -        -   11.50   
                    EQTY        50.00    49.50   0.50    1.00%   50.00   
                    EVRD         0.90     0.93   0.03    3.33%    0.93   
2025-07-19 15:25:06 FTGH         1.33     1.27   0.06    4.51%    1.30   
                    GLD       3900.00  3900.00      -        -     NaN   
2025-07-19 15:25:05 HAFR         0.65     0.68   0.03    4.62%    0.68   
2025-07-19 15:25:06 HBE          4.66     4.66      -        -     NaN   
2025-07-19 15:25:05 HFCK         7.50     7.72   0.22    2.93%    7.80   
2025-07-19 15:25:03 HFCK-R        NaN      NaN      -        -     NaN   
2025-07-19 15:25:06 IMH         36.55    36.50   0.05    0.14%   36.50   
2025-07-19 15:25:05 JUB        235.00   235.00      -        -     NaN   
                    KAPC       340.75   321.75  19.00    5.58%  340.00   
                    KCB         45.90    46.20   0.30    0.65%   46.50   
                    KEGN         7.12     7.16   0.04    0.56%    7.40   
                    KNRE         2.14     2.21   0.07    3.27%    2.29   
                    KPLC        10.70    10.70      -        -   10.95   
2025-07-19 15:25:06 KPLC-P4      4.10     4.10      -        -     NaN   
                    KPLC-P7      6.00     6.00      -        -     NaN   
2025-07-19 15:25:05 KQ           5.28     5.30   0.02    0.38%    5.36   
                    KUKZ       400.00   400.00      -        -     NaN   
                    KURV      1500.00  1500.00      -        -     NaN   
2025-07-19 15:25:06 LAPR        20.00    20.00      -        -     NaN   
                    LBTY        10.80    10.85   0.05    0.46%   10.90   
2025-07-19 15:25:05 LIMT       310.00   310.00      -        -     NaN   
                    LKL          2.54     2.53   0.01    0.39%    2.58   
2025-07-19 15:25:06 MSC          0.27     0.27      -        -     NaN   
2025-07-19 15:25:03 NBK          4.12     4.12      -        -     NaN   
2025-07-19 15:25:05 NBV          1.82     1.81   0.01    0.55%    1.85   
2025-07-19 15:25:06 NCBA        63.00    63.00      -        -   63.50   
2025-07-19 15:25:05 NMG         13.85    13.80   0.05    0.36%   13.95   
                    NSE          9.66     9.52   0.14    1.45%    9.70   
                    OCH          4.40     4.01   0.39    8.86%    4.01   
                    PORT        47.65    47.15   0.50    1.05%   52.25   
                    SASN        15.50    15.55   0.05    0.32%   15.70   
2025-07-19 15:25:06 SBIC       171.25

## 3. Aggregate to monthly level and filter top 150 most liquid stocks for each month.

* To reduce training time and experiment with features and strategies, we convert the business-daily data to month-end frequency.

In [29]:
last_cols = [c for c in df.columns.unique(0) if c not in ['dollar_volume', 'volume', 'open',
                                                          'high', 'low', 'close']]

data = (pd.concat([df.unstack('Security')['dollar_volume'].resample('M').mean().stack('Security').to_frame('dollar_volume'),
                   df.unstack()[last_cols].resample('M').last().stack('Security')],
                  axis=1)).dropna()

data

,,dollar_volume,closing,change,change.1,vwap,deals,turnover,foreign,garman_klass_vol,garman_klass_var,rsi,atr,macd,bb_low,bb_mid,bb_high
Time,Security,,,,,,,,,,,,,,,,


* Calculate 5-year rolling average of dollar volume for each stocks before filtering.

In [31]:
data['dollar_volume'] = (data.loc[:, 'dollar_volume'].unstack('Security').rolling(5*12, min_periods=12).mean().stack())

data['dollar_vol_rank'] = (data.groupby('Time')['dollar_volume'].rank(ascending=False))

data = data[data['dollar_vol_rank']<150].drop(['dollar_volume', 'dollar_vol_rank'], axis=1)

data

,,closing,change,change.1,vwap,deals,turnover,foreign,garman_klass_vol,garman_klass_var,rsi,atr,macd,bb_low,bb_mid,bb_high
Time,Security,,,,,,,,,,,,,,,


## 4. Calculate Monthly Returns for different time horizons as features.

* To capture time series dynamics that reflect, for example, momentum patterns, we compute historical returns using the method .pct_change(lag), that is, returns over various monthly periods as identified by lags.

In [32]:
def calculate_returns(df):

    outlier_cutoff = 0.005

    lags = [1, 2, 3, 6, 9, 12]

    for lag in lags:

        df[f'return_{lag}m'] = (df['closing']
                              .pct_change(lag)
                              .pipe(lambda x: x.clip(lower=x.quantile(outlier_cutoff),
                                                     upper=x.quantile(1-outlier_cutoff)))
                              .add(1)
                              .pow(1/lag)
                              .sub(1))
    return df


data = data.groupby(level=1, group_keys=False).apply(calculate_returns).dropna()

data

,closing,change,change.1,vwap,deals,turnover,foreign,garman_klass_vol,garman_klass_var,rsi,atr,macd,bb_low,bb_mid,bb_high
Security,,,,,,,,,,,,,,,


## 5. Download Fama-French Factors and Calculate Rolling Factor Betas.

* We will introduce the Fama—French data to estimate the exposure of assets to common risk factors using linear regression.

* The five Fama—French factors, namely market risk, size, value, operating profitability, and investment have been shown empirically to explain asset returns and are commonly used to assess the risk/return profile of portfolios. Hence, it is natural to include past factor exposures as financial features in models.

* We can access the historical factor returns using the pandas-datareader and estimate historical exposures using the RollingOLS rolling linear regression.

In [33]:
factor_data = web.DataReader('F-F_Research_Data_5_Factors_2x3',
                               'famafrench',
                               start='2010')[0].drop('RF', axis=1)

factor_data.index = factor_data.index.to_timestamp()

factor_data = factor_data.resample('M').last().div(100)

factor_data.index.name = 'date'

factor_data = factor_data.join(data['return_1m']).sort_index()

factor_data

KeyError: 'return_1m'

* Filter out stocks with less than 10 months of data.

In [ ]:
observations = factor_data.groupby(level=1).size()

valid_stocks = observations[observations >= 10]

factor_data = factor_data[factor_data.index.get_level_values('ticker').isin(valid_stocks.index)]

factor_data

* Calculate Rolling Factor Betas.

In [ ]:
betas = (factor_data.groupby(level=1,
                            group_keys=False)
         .apply(lambda x: RollingOLS(endog=x['return_1m'],
                                     exog=sm.add_constant(x.drop('return_1m', axis=1)),
                                     window=min(24, x.shape[0]),
                                     min_nobs=len(x.columns)+1)
         .fit(params_only=True)
         .params
         .drop('const', axis=1)))

betas

* Join the rolling factors data to the main features dataframe.

In [ ]:
factors = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']

data = (data.join(betas.groupby('ticker').shift()))

data.loc[:, factors] = data.groupby('ticker', group_keys=False)[factors].apply(lambda x: x.fillna(x.mean()))

data = data.drop('closing', axis=1)

data = data.dropna()

data.info()

### At this point we have to decide on what ML model and approach to use for predictions etc.


## 6. For each month fit a K-Means Clustering Algorithm to group similar assets based on their features.

### K-Means Clustering
* You may want to initialize predefined centroids for each cluster based on your research.

* For visualization purpose of this tutorial we will initially rely on the ‘k-means++’ initialization.

* Then we will pre-define our centroids for each cluster.

In [ ]:
from sklearn.cluster import KMeans

data = data.drop('cluster', axis=1)

def get_clusters(df):
    df['cluster'] = KMeans(n_clusters=4,
                           random_state=0,
                           init=initial_centroids).fit(df).labels_
    return df

data = data.dropna().groupby('date', group_keys=False).apply(get_clusters)

data

In [ ]:
def plot_clusters(data):

    cluster_0 = data[data['cluster']==0]
    cluster_1 = data[data['cluster']==1]
    cluster_2 = data[data['cluster']==2]
    cluster_3 = data[data['cluster']==3]

    plt.scatter(cluster_0.iloc[:,0] , cluster_0.iloc[:,6] , color = 'red', label='cluster 0')
    plt.scatter(cluster_1.iloc[:,0] , cluster_1.iloc[:,6] , color = 'green', label='cluster 1')
    plt.scatter(cluster_2.iloc[:,0] , cluster_2.iloc[:,6] , color = 'blue', label='cluster 2')
    plt.scatter(cluster_3.iloc[:,0] , cluster_3.iloc[:,6] , color = 'black', label='cluster 3')

    plt.legend()
    plt.show()
    return


In [ ]:
plt.style.use('ggplot')

for i in data.index.get_level_values('date').unique().tolist():

    g = data.xs(i, level=0)

    plt.title(f'Date {i}')

    plot_clusters(g)

### Apply pre-defined centroids.

In [ ]:
target_rsi_values = [30, 45, 55, 70]

initial_centroids = np.zeros((len(target_rsi_values), 18))

initial_centroids[:, 6] = target_rsi_values

initial_centroids

## 7. For each month select assets based on the cluster and form a portfolio based on Efficient Frontier max sharpe ratio optimization

* First we will filter only stocks corresponding to the cluster we choose based on our hypothesis.

* Momentum is persistent and my idea would be that stocks clustered around RSI 70 centroid should continue to outperform in the following month - thus I would select stocks corresponding to cluster 3.


In [ ]:
filtered_df = data[data['cluster']==3].copy()

filtered_df = filtered_df.reset_index(level=1)

filtered_df.index = filtered_df.index+pd.DateOffset(1)

filtered_df = filtered_df.reset_index().set_index(['date', 'ticker'])

dates = filtered_df.index.get_level_values('date').unique().tolist()

fixed_dates = {}

for d in dates:

    fixed_dates[d.strftime('%Y-%m-%d')] = filtered_df.xs(d, level=0).index.tolist()

fixed_dates

### Define portfolio optimization function

* We will define a function which optimizes portfolio weights using PyPortfolioOpt package and EfficientFrontier optimizer to maximize the sharpe ratio.

* To optimize the weights of a given portfolio we would need to supply last 1 year prices to the function.

* Apply signle stock weight bounds constraint for diversification (minimum half of equaly weight and maximum 10% of portfolio).

In [ ]:
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models
from pypfopt import expected_returns

def optimize_weights(prices, lower_bound=0):

    returns = expected_returns.mean_historical_return(prices=prices,
                                                      frequency=252)

    cov = risk_models.sample_cov(prices=prices,
                                 frequency=252)

    ef = EfficientFrontier(expected_returns=returns,
                           cov_matrix=cov,
                           weight_bounds=(lower_bound, .1),
                           solver='SCS')

    weights = ef.max_sharpe()

    return ef.clean_weights()


* Download Fresh Daily Prices Data only for short listed stocks.

In [ ]:
stocks = data.index.get_level_values('ticker').unique().tolist()

new_df = yf.download(tickers=stocks,
                     start=data.index.get_level_values('date').unique()[0]-pd.DateOffset(months=12),
                     end=data.index.get_level_values('date').unique()[-1])

new_df

* Calculate daily returns for each stock which could land up in our portfolio.

* Then loop over each month start, select the stocks for the month and calculate their weights for the next month.

* If the maximum sharpe ratio optimization fails for a given month, apply equally-weighted weights.

* Calculated each day portfolio return.

In [ ]:
returns_dataframe = np.log(new_df['closing']).diff()

portfolio_df = pd.DataFrame()

for start_date in fixed_dates.keys():

    try:

        end_date = (pd.to_datetime(start_date)+pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

        cols = fixed_dates[start_date]

        optimization_start_date = (pd.to_datetime(start_date)-pd.DateOffset(months=12)).strftime('%Y-%m-%d')

        optimization_end_date = (pd.to_datetime(start_date)-pd.DateOffset(days=1)).strftime('%Y-%m-%d')

        optimization_df = new_df[optimization_start_date:optimization_end_date]['closing'][cols]

        success = False
        try:
            weights = optimize_weights(prices=optimization_df,
                                   lower_bound=round(1/(len(optimization_df.columns)*2),3))

            weights = pd.DataFrame(weights, index=pd.Series(0))

            success = True
        except:
            print(f'Max Sharpe Optimization failed for {start_date}, Continuing with Equal-Weights')

        if success==False:
            weights = pd.DataFrame([1/len(optimization_df.columns) for i in range(len(optimization_df.columns))],
                                     index=optimization_df.columns.tolist(),
                                     columns=pd.Series(0)).T

        temp_df = returns_dataframe[start_date:end_date]

        temp_df = temp_df.stack().to_frame('return').reset_index(level=0)\
                   .merge(weights.stack().to_frame('weight').reset_index(level=0, drop=True),
                          left_index=True,
                          right_index=True)\
                   .reset_index().set_index(['Date', 'index']).unstack().stack()

        temp_df.index.names = ['date', 'ticker']

        temp_df['weighted_return'] = temp_df['return']*temp_df['weight']

        temp_df = temp_df.groupby(level=0)['weighted_return'].sum().to_frame('Strategy Return')

        portfolio_df = pd.concat([portfolio_df, temp_df], axis=0)

    except Exception as e:
        print(e)

portfolio_df = portfolio_df.drop_duplicates()

portfolio_df

## 8. Visualize Portfolio returns and compare to SP500 returns.

In [ ]:
spy = yf.download(tickers='SPY',
                  start='2015-01-01',
                  end=dt.date.today())

spy_ret = np.log(spy[['closing']]).diff().dropna().rename({'closing':'SPY Buy&Hold'}, axis=1)

portfolio_df = portfolio_df.merge(spy_ret,
                                  left_index=True,
                                  right_index=True)

portfolio_df

In [ ]:
import matplotlib.ticker as mtick

plt.style.use('ggplot')

portfolio_cumulative_return = np.exp(np.log1p(portfolio_df).cumsum())-1

portfolio_cumulative_return[:'2023-09-29'].plot(figsize=(16,6))

plt.title('Unsupervised Learning Trading Strategy Returns Over Time')

plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1))

plt.ylabel('Return')

plt.show()


# Twitter Sentiment Investing Strategy

## 1. Load Twitter Sentiment Data

* Load the twitter sentiment dataset, set the index, calculat engagement ratio and filter out stocks with no significant twitter activity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
import yfinance as yf
import os
plt.style.use('ggplot')

data_folder = 'C:/Users/user/Desktop/Python Scripts'

sentiment_df = pd.read_csv(os.path.join(data_folder, 'sentiment_data.csv'))

sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])

sentiment_df = sentiment_df.set_index(['date', 'symbol'])

sentiment_df['engagement_ratio'] = sentiment_df['twitterComments']/sentiment_df['twitterLikes']

sentiment_df = sentiment_df[(sentiment_df['twitterLikes']>20)&(sentiment_df['twitterComments']>10)]

sentiment_df

## 2. Aggregate Monthly and calculate average sentiment for the month

* Aggregate on a monthly level and calculate average monthly metric, for the one we choose.

In [ ]:
aggragated_df = (sentiment_df.reset_index('symbol').groupby([pd.Grouper(freq='M'), 'symbol'])
                    [['engagement_ratio']].mean())

aggragated_df['rank'] = (aggragated_df.groupby(level=0)['engagement_ratio']
                         .transform(lambda x: x.rank(ascending=False)))

aggragated_df

## 3. Select Top 5 Stocks based on their cross-sectional ranking for each month

* Select top 5 stocks by rank for each month and fix the date to start at beginning of next month.

In [ ]:
filtered_df = aggragated_df[aggragated_df['rank']<6].copy()

filtered_df = filtered_df.reset_index(level=1)

filtered_df.index = filtered_df.index+pd.DateOffset(1)

filtered_df = filtered_df.reset_index().set_index(['date', 'symbol'])

filtered_df.head(20)

## 4. Extract the stocks to form portfolios with at the start of each new month

* Create a dictionary containing start of month and corresponded selected stocks.

In [ ]:
dates = filtered_df.index.get_level_values('date').unique().tolist()

fixed_dates = {}

for d in dates:

    fixed_dates[d.strftime('%Y-%m-%d')] = filtered_df.xs(d, level=0).index.tolist()

fixed_dates

## 5. Download fresh stock prices for only selected/shortlisted stocks

In [ ]:
stocks_list = sentiment_df.index.get_level_values('symbol').unique().tolist()

prices_df = yf.download(tickers=stocks_list,
                        start='2021-01-01',
                        end='2023-03-01')

## 6. Calculate Portfolio Returns with monthly rebalancing


In [ ]:
returns_df = np.log(prices_df['closing']).diff().dropna()

portfolio_df = pd.DataFrame()

for start_date in fixed_dates.keys():

    end_date = (pd.to_datetime(start_date)+pd.offsets.MonthEnd()).strftime('%Y-%m-%d')

    cols = fixed_dates[start_date]

    temp_df = returns_df[start_date:end_date][cols].mean(axis=1).to_frame('portfolio_return')

    portfolio_df = pd.concat([portfolio_df, temp_df], axis=0)

portfolio_df

## 7. Download NASDAQ/QQQ prices and calculate returns to compare to our strategy

In [ ]:
qqq_df = yf.download(tickers='QQQ',
                     start='2021-01-01',
                     end='2023-03-01')

qqq_ret = np.log(qqq_df['closing']).diff().to_frame('nasdaq_return')

portfolio_df = portfolio_df.merge(qqq_ret,
                                  left_index=True,
                                  right_index=True)

portfolio_df

In [ ]:
portfolios_cumulative_return = np.exp(np.log1p(portfolio_df).cumsum()).sub(1)

portfolios_cumulative_return.plot(figsize=(16,6))

plt.title('Twitter Engagement Ratio Strategy Return Over Time')

plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1))

plt.ylabel('Return')

plt.show()

# Intraday Strategy Using GARCH Model


* Using simulated daily data and intraday 5-min data.
* Load Daily and 5-minute data.
* Define function to fit GARCH model on the daily data and predict 1-day ahead volatility in a rolling window.
* Calculate prediction premium and form a daily signal from it.
* Merge with intraday data and calculate intraday indicators to form the intraday signal.
* Generate the position entry and hold until the end of the day.
* Calculate final strategy returns.

## 1. Load Simulated Daily and Simulated 5-minute data.

* We are loading both datasets, set the indexes and calculate daily log returns.

In [ ]:
import matplotlib.pyplot as plt
from arch import arch_model
import pandas_ta
import pandas as pd
import numpy as np
import os

data_folder = 'C:/Users/user/Desktop/Python Scripts'

daily_df = pd.read_csv(os.path.join(data_folder, 'simulated_daily_data.csv'))

daily_df = daily_df.drop('Unnamed: 7', axis=1)

daily_df['Date'] = pd.to_datetime(daily_df['Date'])

daily_df = daily_df.set_index('Date')


intraday_5min_df = pd.read_csv(os.path.join(data_folder, 'simulated_5min_data.csv'))

intraday_5min_df = intraday_5min_df.drop('Unnamed: 6', axis=1)

intraday_5min_df['datetime'] = pd.to_datetime(intraday_5min_df['datetime'])

intraday_5min_df = intraday_5min_df.set_index('datetime')

intraday_5min_df['date'] = pd.to_datetime(intraday_5min_df.index.date)

intraday_5min_df

## 2. Define function to fit GARCH model and predict 1-day ahead volatility in a rolling window.

* We are first calculating the 6-month rolling variance and then we are creating a function in a 6-month rolling window to fit a garch model and predict the next day variance.

In [ ]:
daily_df['log_ret'] = np.log(daily_df['closing']).diff()

daily_df['variance'] = daily_df['log_ret'].rolling(180).var()

daily_df = daily_df['2020':]

def predict_volatility(x):

    best_model = arch_model(y=x,
                            p=1,
                            q=3).fit(update_freq=5,
                                     disp='off')

    variance_forecast = best_model.forecast(horizon=1).variance.iloc[-1,0]

    print(x.index[-1])

    return variance_forecast

daily_df['predictions'] = daily_df['log_ret'].rolling(180).apply(lambda x: predict_volatility(x))

daily_df = daily_df.dropna()

daily_df

## 3. Calculate prediction premium and form a daily signal from it.

* We are calculating the prediction premium. And calculate its 6-month rolling standard deviation.

* From this we are creating our daily signal.

In [ ]:
daily_df['prediction_premium'] = (daily_df['predictions']-daily_df['variance'])/daily_df['variance']

daily_df['premium_std'] = daily_df['prediction_premium'].rolling(180).std()

daily_df['signal_daily'] = daily_df.apply(lambda x: 1 if (x['prediction_premium']>x['premium_std'])
                                         else (-1 if (x['prediction_premium']<x['premium_std']*-1) else np.nan),
                                         axis=1)

daily_df['signal_daily'] = daily_df['signal_daily'].shift()

daily_df

In [ ]:
plt.style.use('ggplot')

daily_df['signal_daily'].plot(kind='hist')

plt.show()

## 4. Merge with intraday data and calculate intraday indicators to form the intraday signal.

* Calculate all intraday indicators and intraday signal.

In [ ]:
final_df = intraday_5min_df.reset_index()\
                            .merge(daily_df[['signal_daily']].reset_index(),
                                   left_on='date',
                                   right_on='Date')\
                            .drop(['date','Date'], axis=1)\
                            .set_index('datetime')

final_df['rsi'] = pandas_ta.rsi(close=final_df['close'],
                                length=20)

final_df['lband'] = pandas_ta.bbands(close=final_df['close'],
                                     length=20).iloc[:,0]

final_df['uband'] = pandas_ta.bbands(close=final_df['close'],
                                     length=20).iloc[:,2]

final_df['signal_intraday'] = final_df.apply(lambda x: 1 if (x['rsi']>70)&
                                                            (x['close']>x['uband'])
                                             else (-1 if (x['rsi']<30)&
                                                         (x['close']<x['lband']) else np.nan),
                                             axis=1)

final_df['return'] = np.log(final_df['close']).diff()

final_df

## 5. Generate the position entry and hold until the end of the day.

In [ ]:
final_df['return_sign'] = final_df.apply(lambda x: -1 if (x['signal_daily']==1)&(x['signal_intraday']==1)
                                        else (1 if (x['signal_daily']==-1)&(x['signal_intraday']==-1) else np.nan),
                                        axis=1)

final_df['return_sign'] = final_df.groupby(pd.Grouper(freq='D'))['return_sign']\
                                  .transform(lambda x: x.ffill())

final_df['forward_return'] = final_df['return'].shift(-1)

final_df['strategy_return'] = final_df['forward_return']*final_df['return_sign']

daily_return_df = final_df.groupby(pd.Grouper(freq='D'))['strategy_return'].sum()

## 6. Calculate final strategy returns.

In [ ]:
import matplotlib.ticker as mtick

strategy_cumulative_return = np.exp(np.log1p(daily_return_df).cumsum()).sub(1)

strategy_cumulative_return.plot(figsize=(16,6))

plt.title('Intraday Strategy Returns')

plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1))

plt.ylabel('Return')

plt.show()
